# Insulation score и boundaries для Hi-C матриц

Блокнот с **insulation score** и **TAD boundaries**.

insulation score показывает, насколько хорошо участок генома изолирует контакты слева и справа от себя. Изоляция компартментов друг от друга происходит на границах доменов


In [ ]:
import os
import itertools
import warnings
warnings.filterwarnings("ignore")

import cooler
import cooltools
import bioframe

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import LogNorm
from matplotlib.ticker import EngFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable

from cooltools import insulation

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 11


## 1. Загрузка данных

Для insulation analysis обычно используют более высокое разрешение, чем для компартметров: 10kb, 25kb, 50kb или 100kb.


In [ ]:
resolution = 100_000

sample_names = ['MoPh7_enr_v2', 'MoPh11_enr_v2', 'MoPh14_enr_v2', 'MoPh15_enr_v2']

cool_paths = {
    sample: f'../data/{sample}.mcool::/resolutions/{resolution}'
    for sample in sample_names
}

clrs = {
    sample: cooler.Cooler(path)
    for sample, path in cool_paths.items()
}


## 2. Каноничные хромосомы


In [ ]:
# Берём первый образец как референс для списка хромосом, оставляем только каноничные хромосомы для анализа
example_clr = clrs['MoPh7_enr_v2']

canonical_chroms = [str(i) for i in range(1, 23)] + ['X', 'Y']

chromsizes = example_clr.chromsizes.reset_index()
chromsizes.columns = ['chrom', 'length']

chromsizes = chromsizes[
    chromsizes['chrom'].isin(canonical_chroms)
].copy()

view_df = pd.DataFrame({
    'chrom': chromsizes['chrom'],
    'start': 0,
    'end': chromsizes['length'],
    'name': chromsizes['chrom']
})

view_df.head()


## 3. Создаём view_df

`view_df` задаёт регионы, в пределах которых будет считаться insulation score.

Здесь мы используем целые canonical chromosomes.


In [ ]:
chromsizes = example_clr.chromsizes.reset_index()
chromsizes.columns = ['chrom', 'length']

chromsizes = chromsizes[
    chromsizes['chrom'].isin(canonical_chroms)
].copy()

view_df = pd.DataFrame({
    'chrom': chromsizes['chrom'],
    'start': 0,
    'end': chromsizes['length'],
    'name': chromsizes['chrom']
})

view_df.head()


## 4. Выбираем размеры окон

Insulation score считается в diamond-shaped window около диагонали Hi-C матрицы
Размер окна влияет на коллинг границ:

- маленькое окно видит локальные границы
- большое окно видит более крупные домены
- слишком большое окно на грубом разрешении может сильно сгладить


In [ ]:
windows = [
    3 * resolution,
    5 * resolution,
    10 * resolution,
]

print('Resolution:', resolution)
print('Windows:', windows)


## 5. Считаем insulation score для одного образца

`cooltools.insulation` возвращает таблицу, где каждая строка соответствует бину генома.

Для каждого размера окна появляются колонки:

- `log2_insulation_score_WINDOW`
- `n_valid_pixels_WINDOW`
- `boundary_strength_WINDOW`
- `is_boundary_WINDOW`


In [ ]:
sample = 'MoPh7_enr_v2'
clr = clrs[sample]

if 'weight' not in clr.bins().columns:
    raise ValueError('This cooler has no weight column. Please run cooler balance first.')

insulation_table = insulation(
    clr,
    windows,
    view_df=view_df,
    clr_weight_name='weight',
    nproc=4
)

insulation_table.head()


In [ ]:
largest_window = windows[-1]

window_columns = [
    col for col in insulation_table.columns
    if str(largest_window) in col
]

insulation_table[
    ['chrom', 'start', 'end', 'region', 'is_bad_bin'] + window_columns
].head()


## 6. Вспомогательные функции для отрисовки

Повернем Hi-C матрицу на 45 градусов для графика


In [ ]:
def pcolormesh_45deg(ax, matrix_c, start=0, resolution=1, *args, **kwargs):
    start_pos_vector = [start + resolution * i for i in range(len(matrix_c) + 1)]
    n = matrix_c.shape[0]

    transform = np.array([
        [1, 0.5],
        [-1, 0.5]
    ])

    matrix_a = np.dot(
        np.array([
            (i[1], i[0])
            for i in itertools.product(start_pos_vector[::-1], start_pos_vector)
        ]),
        transform
    )

    x = matrix_a[:, 1].reshape(n + 1, n + 1)
    y = matrix_a[:, 0].reshape(n + 1, n + 1)

    im = ax.pcolormesh(x, y, np.flipud(matrix_c), *args, **kwargs)
    im.set_rasterized(True)

    return im


bp_formatter = EngFormatter('b')


def format_ticks(ax, x=True, y=True, rotate=True):
    if y:
        ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
        ax.xaxis.tick_bottom()
    if rotate:
        ax.tick_params(axis='x', rotation=45)


## 7. Выбираем регион для визуализации


In [ ]:
chrom = '2'

if chrom not in canonical_chroms:
    chrom = canonical_chroms[0]

chrom_length = int(chromsizes.loc[chromsizes['chrom'] == chrom, 'length'].iloc[0])
start = 10_000_000
end = start + 30 * windows[0]
region = (chrom, start, end)
print('Region:', region)


## 8. Визуализация


In [ ]:
window = windows[0]

matrix = clr.matrix(balance=True).fetch(region)
insul_region = bioframe.select(insulation_table, region)

positive_values = matrix[np.isfinite(matrix) & (matrix > 0)]
vmax = np.nanpercentile(positive_values, 99) if len(positive_values) > 0 else 1
vmin = np.nanpercentile(positive_values, 5) if len(positive_values) > 0 else 1e-4

fig, ax = plt.subplots(figsize=(14, 6))

im = pcolormesh_45deg(
    ax,
    matrix,
    start=region[1],
    resolution=resolution,
    norm=LogNorm(vmin=vmin, vmax=vmax)
)

ax.set_aspect(0.5)
ax.set_ylim(0, 10 * window)
format_ticks(ax, rotate=False)
ax.xaxis.set_visible(False)

divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='1%', pad=0.1, aspect=6)
plt.colorbar(im, cax=cax, label='balanced contacts')

ins_ax = divider.append_axes('bottom', size='50%', pad=0., sharex=ax)

ins_ax.plot(
    insul_region[['start', 'end']].mean(axis=1),
    insul_region[f'log2_insulation_score_{window}'],
    label=f'Window {window:,} bp'
)

ins_ax.set(
    ylabel='log2 insulation score',
    xlabel=f'{chrom} position'
)

format_ticks(ins_ax, y=False, rotate=False)
ins_ax.legend(loc='lower left')

ax.set_xlim(region[1], region[2])
ax.set_title(f'{sample}: Hi-C matrix and insulation score')

plt.show()


## 9. Добавляем на график несколько window sizes

Посмотрим на инсуляцию для всех выбранных окон. Увилим, что результат зависит от масштаба окна.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

for window in windows:
    ax.plot(
        insul_region[['start', 'end']].mean(axis=1),
        insul_region[f'log2_insulation_score_{window}'],
        label=f'{window // 1000} kb'
    )

ax.set(
    xlabel=f'{chrom} position',
    ylabel='log2 insulation score',
    title=f'{sample}: insulation profiles for different window sizes'
)

format_ticks(ax, y=False, rotate=False)
ax.legend(title='Window')

plt.show()


## 10. Коллинг границ доменов

`cooltools.insulation` автоматически находит потенциальные границы и оценивает их силу

Для каждого окна есть:

- `boundary_strength_WINDOW`
- `is_boundary_WINDOW`

`is_boundary` — рассчитывается функцией автоматически


In [ ]:
window = windows[0]

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(
    insul_region[['start', 'end']].mean(axis=1),
    insul_region[f'log2_insulation_score_{window}'],
    label=f'Insulation score, window {window // 1000} kb'
)

boundaries = insul_region[
    ~insul_region[f'boundary_strength_{window}'].isna()
].copy()

weak_boundaries = boundaries[
    ~boundaries[f'is_boundary_{window}']
]

strong_boundaries = boundaries[
    boundaries[f'is_boundary_{window}']
]

ax.scatter(
    weak_boundaries[['start', 'end']].mean(axis=1),
    weak_boundaries[f'log2_insulation_score_{window}'],
    s=20,
    label='Weak boundaries'
)

ax.scatter(
    strong_boundaries[['start', 'end']].mean(axis=1),
    strong_boundaries[f'log2_insulation_score_{window}'],
    s=30,
    label='Strong boundaries'
)

ax.set(
    xlabel=f'{chrom} position',
    ylabel='log2 insulation score',
    title=f'{sample}: boundaries on insulation profile'
)

format_ticks(ax, y=False, rotate=False)
ax.legend()

plt.show()


## Задание для самостоятельного разбора

Сохраните границы доменов в формате `bedGraph`, чтобы позже пересечь их с CTCF-треком на занятии по ChIP-seq.

Что нужно сделать:

1. Взять таблицу `insulation_table`.
2. Выбрать строки, где `is_boundary_WINDOW == True` для одного из window sizes.
3. Оставить колонки `chrom`, `start`, `end` и `boundary_strength_WINDOW`.
4. Сохранить файл в папку `../results/boundaries/`.

Ожидаемый результат:

```text
results/boundaries/MoPh7_enr_v2_boundaries_100kb.bedGraph
```

Подсказка: для `bedGraph` не нужна строка с названиями колонок, поэтому в `to_csv()` можно использовать `header=False`.
